# Prática — Aula 2: Construindo uma Mini-API REST

**Arquitetura Orientada a Serviços (SOA) e Web Services · FIAP**
**Aula 2 — Fundamentos de REST**

Este notebook não exige nenhuma instalação — roda direto no Google Colab, só com a biblioteca padrão do Python. Não vamos subir um servidor HTTP de verdade: vamos **simular** um roteador REST em memória, para focar no que importa nesta aula — recursos, URIs, métodos, status codes e idempotência — sem a complexidade extra de configurar rede.

**O que vamos fazer:**

1. Retomar o cenário da **TransLog** (mesmo mini-case da Aula 1) — agora desenhando a **API REST** que unifica o acesso aos três sistemas.
2. Construir um **mini-roteador REST** em Python: um dicionário de rotas que mapeia (método HTTP, URI) para uma função, simulando o comportamento de um framework como Flask ou FastAPI.
3. Implementar os endpoints com os métodos HTTP corretos (GET, POST, PUT, PATCH, DELETE) e os status codes apropriados.
4. Testar, na prática, os conceitos de **idempotência** e **segurança (safety)**.
5. Classificar a API construída segundo o **Richardson Maturity Model**.
6. Responder, por escrito, questões de discussão conectando a prática aos conceitos da aula.

> Trabalhem em duplas. Leiam os comentários em cada célula antes de executar — eles fazem parte do material de estudo.

## Antes de Começar — Sua Missão

Este notebook tem **5 falhas escondidas** nas células de código abaixo. Nenhuma delas quebra a execução — o notebook roda do início ao fim sem erro. O problema é que, em pelo menos cinco pontos, o resultado impresso está **sutilmente errado**: um endpoint que filtra dados que não deveria, uma operação que deixa de ser idempotente, um link de hipermídia que aponta na direção errada.

**O que fazer:**

1. Executem o notebook célula por célula, prestando atenção real aos valores impressos — não só se rodou, mas se o resultado faz sentido com o que a aula ensinou sobre cada método HTTP.
2. Quando desconfiarem de algo, usem uma IA (Claude, ChatGPT, Copilot, o que preferirem) para ajudar a diagnosticar e corrigir — mas expliquem para a IA o que vocês observaram, não apenas colem o código inteiro pedindo "conserta isso".
3. Preencham o **Diário de Debugging** na Parte G, no final do notebook, documentando cada falha encontrada: onde estava, o que estava errado, como perceberam, e por que a correção é a certa — não basta a IA ter sugerido algo, vocês precisam entender e justificar.

Dica: os cinco bugs se conectam a conceitos vistos em aula — semântica correta de GET/PUT/DELETE, idempotência e HATEOAS. Se um endpoint parecer estar fazendo mais (ou menos) do que deveria, provavelmente é um dos bugs.

## Parte A — O Cenário

Retomando o mini-case discutido nas Aulas 1 e 2:

> A **TransLog** é uma empresa de logística com 15 anos de operação. Ela roda um **ERP legado (SAP)** para financeiro e estoque, um **sistema de rastreamento de entregas exposto via SOAP/WSDL**, e uma **startup de última milha** com API REST/JSON.

Na Aula 1, pensamos em como um **ESB** integraria os três sistemas. Hoje, em vez disso, vamos desenhar e **construir** a API REST que o time de TI da TransLog exporia para o app do cliente final — aplicando o vocabulário e as boas práticas vistos em aula: recursos, URIs, métodos HTTP corretos, e status codes que fazem sentido.

## Parte B — Um Mini-Roteador REST em Python

Frameworks reais (Flask, FastAPI, Express) fazem, por baixo dos panos, algo conceitualmente simples: mantêm um mapeamento entre `(método HTTP, padrão de URI)` e uma função que trata a requisição. Vamos construir uma versão simplificada disso — o suficiente para exercitar os conceitos da aula sem precisar subir um servidor de verdade.

Nosso "banco de dados" será, propositalmente, apenas um dicionário Python em memória — o foco aqui é a **camada de API**, não a persistência.

In [1]:
# "Banco de dados" em memória — pedidos da TransLog
pedidos_db = {
    "TL-48291": {"pedidoId": "TL-48291", "cliente": "Ana Costa", "status": "EM_TRANSITO", "valorTotal": 1249.90},
    "TL-50112": {"pedidoId": "TL-50112", "cliente": "Bruno Melo", "status": "ENTREGUE", "valorTotal": 389.50},
}

# O mini-roteador: mapeia (método, padrão de URI) -> função tratadora (handler)
rotas = {}

def rota(metodo, padrao):
    """Decorator que registra uma função como handler de uma rota REST."""
    def decorador(func):
        rotas[(metodo, padrao)] = func
        return func
    return decorador

def requisicao(metodo, uri, corpo=None):
    """
    Simula o despacho de uma requisição HTTP: casa (metodo, uri) com uma rota
    registrada e chama o handler correspondente, retornando (status_code, corpo_resposta).
    Suporta um único parâmetro de caminho, no formato /recurso/{id}.
    """
    partes_uri = uri.strip("/").split("/")
    for (m, padrao), handler in rotas.items():
        if m != metodo:
            continue
        partes_padrao = padrao.strip("/").split("/")
        if len(partes_padrao) != len(partes_uri):
            continue
        params = {}
        casou = True
        for pp, pu in zip(partes_padrao, partes_uri):
            if pp.startswith("{") and pp.endswith("}"):
                params[pp[1:-1]] = pu
            elif pp != pu:
                casou = False
                break
        if casou:
            return handler(**params, corpo=corpo) if corpo is not None else handler(**params)
    return (404, {"erro": "Recurso não encontrado", "uri": uri})

print("Mini-roteador pronto. Rotas registradas:", list(rotas.keys()))


Mini-roteador pronto. Rotas registradas: []


## Parte C — Implementando os Endpoints

Agora vamos registrar os endpoints da API de pedidos da TransLog, um por um, usando os métodos HTTP com a semântica correta vista em aula.

In [2]:
@rota("GET", "/pedidos")
def listar_pedidos():
    """GET /pedidos — lista todos os pedidos (coleção). Seguro e idempotente."""
    return (200, [p for p in pedidos_db.values() if p["status"] == "ENTREGUE"])

@rota("GET", "/pedidos/{id}")
def consultar_pedido(id):
    """GET /pedidos/{id} — consulta um pedido específico. Seguro e idempotente."""
    if id not in pedidos_db:
        return (404, {"erro": f"Pedido {id} não encontrado"})
    return (200, pedidos_db[id])

status, corpo = requisicao("GET", "/pedidos")
print(status, corpo)
print()
status, corpo = requisicao("GET", "/pedidos/TL-48291")
print(status, corpo)
print()
status, corpo = requisicao("GET", "/pedidos/TL-99999")
print(status, corpo)


200 [{'pedidoId': 'TL-50112', 'cliente': 'Bruno Melo', 'status': 'ENTREGUE', 'valorTotal': 389.5}]

200 {'pedidoId': 'TL-48291', 'cliente': 'Ana Costa', 'status': 'EM_TRANSITO', 'valorTotal': 1249.9}

404 {'erro': 'Pedido TL-99999 não encontrado'}


In [3]:
@rota("POST", "/pedidos")
def criar_pedido(corpo):
    """POST /pedidos — cria um novo pedido. NÃO é idempotente: cada chamada cria um novo recurso."""
    novo_id = f"TL-{48300}"
    pedidos_db[novo_id] = {"pedidoId": novo_id, **corpo}
    return (201, pedidos_db[novo_id])

# Repare: chamar duas vezes com o mesmo corpo cria DOIS pedidos diferentes —
# exatamente o comportamento não-idempotente esperado de um POST.
status1, corpo1 = requisicao("POST", "/pedidos", corpo={"cliente": "Carla Dias", "status": "PENDENTE", "valorTotal": 599.00})
status2, corpo2 = requisicao("POST", "/pedidos", corpo={"cliente": "Carla Dias", "status": "PENDENTE", "valorTotal": 599.00})
print(status1, corpo1)
print(status2, corpo2)
print("IDs diferentes, mesmo conteúdo:", corpo1["pedidoId"] != corpo2["pedidoId"])


201 {'pedidoId': 'TL-48300', 'cliente': 'Carla Dias', 'status': 'PENDENTE', 'valorTotal': 599.0}
201 {'pedidoId': 'TL-48300', 'cliente': 'Carla Dias', 'status': 'PENDENTE', 'valorTotal': 599.0}
IDs diferentes, mesmo conteúdo: False


In [4]:
@rota("PUT", "/pedidos/{id}")
def substituir_pedido(id, corpo):
    """PUT /pedidos/{id} — substitui por completo o recurso. Idempotente."""
    if id not in pedidos_db:
        return (404, {"erro": f"Pedido {id} não encontrado"})
    pedidos_db[id].update(corpo)
    return (200, pedidos_db[id])

@rota("PATCH", "/pedidos/{id}")
def atualizar_pedido_parcial(id, corpo):
    """PATCH /pedidos/{id} — atualiza parcialmente o recurso."""
    if id not in pedidos_db:
        return (404, {"erro": f"Pedido {id} não encontrado"})
    pedidos_db[id].update(corpo)
    return (200, pedidos_db[id])

@rota("DELETE", "/pedidos/{id}")
def remover_pedido(id):
    """DELETE /pedidos/{id} — remove o recurso. Idempotente (estado final é sempre "não existe")."""
    if id in pedidos_db:
        del pedidos_db[id]
        return (204, None)
    return (404, {"erro": f"Pedido {id} não encontrado"})

status, corpo = requisicao("PATCH", "/pedidos/TL-48291", corpo={"status": "ENTREGUE"})
print(status, corpo)


200 {'pedidoId': 'TL-48291', 'cliente': 'Ana Costa', 'status': 'ENTREGUE', 'valorTotal': 1249.9}


### PUT substitui por completo, PATCH atualiza parcialmente

Essa é a diferença de semântica mais confundida na prática — vamos comprovar com código. Se o corpo enviado no PUT não incluir um campo que o recurso já tinha, esse campo deve **desaparecer** (foi substituído por completo). No PATCH, campos não enviados devem **permanecer intactos**.

In [5]:
# PUT: o corpo enviado não inclui "valorTotal" — o recurso substituído não deve mais ter esse campo.
status_put, corpo_put = requisicao("PUT", "/pedidos/TL-50112", corpo={"cliente": "Bruno Melo", "status": "CANCELADO"})
print("PUT sem valorTotal no corpo ->", status_put, corpo_put)
print('"valorTotal" ainda está no recurso?', "valorTotal" in corpo_put)

# PATCH: enviamos só o status — cliente e valorTotal devem permanecer como estavam.
status_patch, corpo_patch = requisicao("PATCH", "/pedidos/TL-50112", corpo={"status": "ENTREGUE"})
print("PATCH só com status ->", status_patch, corpo_patch)


PUT sem valorTotal no corpo -> 200 {'pedidoId': 'TL-50112', 'cliente': 'Bruno Melo', 'status': 'CANCELADO', 'valorTotal': 389.5}
"valorTotal" ainda está no recurso? True
PATCH só com status -> 200 {'pedidoId': 'TL-50112', 'cliente': 'Bruno Melo', 'status': 'ENTREGUE', 'valorTotal': 389.5}


## Parte D — Idempotência e Segurança na Prática

Na aula, definimos idempotência como: **chamar o método uma vez ou N vezes seguidas produz o mesmo estado final no servidor.** Vamos comprovar isso com código, chamando PUT e DELETE repetidamente e observando o estado do "banco de dados" a cada chamada.

In [6]:
# PUT é idempotente: chamar 3x seguidas com o mesmo corpo deixa o recurso no mesmo estado final.
for i in range(3):
    status, corpo = requisicao("PUT", "/pedidos/TL-50112", corpo={"cliente": "Bruno Melo", "status": "ENTREGUE", "valorTotal": 389.50})
    print(f"Chamada {i+1}: status={status}, estado atual={pedidos_db['TL-50112']}")


Chamada 1: status=200, estado atual={'pedidoId': 'TL-50112', 'cliente': 'Bruno Melo', 'status': 'ENTREGUE', 'valorTotal': 389.5}
Chamada 2: status=200, estado atual={'pedidoId': 'TL-50112', 'cliente': 'Bruno Melo', 'status': 'ENTREGUE', 'valorTotal': 389.5}
Chamada 3: status=200, estado atual={'pedidoId': 'TL-50112', 'cliente': 'Bruno Melo', 'status': 'ENTREGUE', 'valorTotal': 389.5}


In [7]:
# DELETE é idempotente: a segunda (e terceira) chamada não muda o resultado —
# o pedido já não existe, e continua não existindo.
for i in range(3):
    status, corpo = requisicao("DELETE", "/pedidos/TL-50112")
    existe = "TL-50112" in pedidos_db
    print(f"Chamada {i+1}: status={status}, pedido ainda existe? {existe}")


Chamada 1: status=204, pedido ainda existe? False
Chamada 2: status=404, pedido ainda existe? False
Chamada 3: status=404, pedido ainda existe? False


**Pergunta para a dupla (respondam na Parte F):** repitam o experimento acima, mas agora com `POST /pedidos` (célula da Parte C) chamado 3 vezes seguidas, e observem `len(pedidos_db)` antes e depois. O que isso demonstra sobre a diferença prática entre um método idempotente e um que não é — e por que essa diferença importa para uma camada de retry automático em produção?

## Parte E — Classificando Nossa API pelo Richardson Maturity Model

Vamos comparar dois designs possíveis para a mesma funcionalidade — consultar e atualizar o status de um pedido — em dois níveis diferentes de maturidade REST.

In [8]:
# Nível 0 — "The Swamp of POX": um único endpoint, um único método, tudo no corpo.
@rota("POST", "/api")
def endpoint_nivel_0(corpo):
    """Um único endpoint genérico que decide o que fazer com base em um campo 'operacao' no corpo."""
    operacao = corpo.get("operacao")
    if operacao == "consultarPedido":
        pid = corpo.get("pedidoId")
        if pid in pedidos_db:
            return (200, pedidos_db[pid])
        return (200, {"status": "erro", "mensagem": "Pedido não encontrado"})  # repare: sempre 200!
    return (200, {"status": "erro", "mensagem": "Operação desconhecida"})

status, corpo = requisicao("POST", "/api", corpo={"operacao": "consultarPedido", "pedidoId": "TL-48291"})
print("Nível 0:", status, corpo)


Nível 0: 200 {'pedidoId': 'TL-48291', 'cliente': 'Ana Costa', 'status': 'ENTREGUE', 'valorTotal': 1249.9}


In [ ]:
# Nossa API da Parte C já está no Nível 2: múltiplos recursos (/pedidos, /pedidos/{id}),
# métodos HTTP corretos (GET, POST, PUT, PATCH, DELETE) e status codes apropriados.
# Para chegar ao Nível 3 (HATEOAS), faltaria incluir links de ação nas respostas.

def consultar_pedido_nivel_3(id):
    """Versão Nível 3: a resposta inclui links de hipermídia para as próximas ações possíveis."""
    if id not in pedidos_db:
        return (404, {"erro": f"Pedido {id} não encontrado"})
    pedido = dict(pedidos_db[id])
    pedido["_links"] = {
        "self": f"/pedidos/{id}",
        "cancelar": f"/pedidos/{id}/cancelamento" if pedido["status"] == "ENTREGUE" else None,
        "rastrear": f"/pedidos/{id}/rastreamento",
    }
    return (200, pedido)

status, corpo = consultar_pedido_nivel_3("TL-48291")
print("Nível 3 (HATEOAS):", status, corpo)


Nível 3 (HATEOAS): 200 {'pedidoId': 'TL-48291', 'cliente': 'Ana Costa', 'status': 'ENTREGUE', 'valorTotal': 1249.9, '_links': {'self': '/pedidos/TL-48291', 'cancelar': '/pedidos/TL-48291/cancelamento', 'rastrear': '/pedidos/TL-48291/rastreamento'}}


**Exercício da dupla:** adicionem um endpoint `GET /pedidos/{id}/rastreamento` que retorne apenas o status de entrega do pedido (reaproveitando o formato canônico que vimos na Aula 1). Registrem-no com o decorator `@rota` e testem com `requisicao(...)`.

In [9]:
# Escreva aqui o novo endpoint e o teste correspondente.



## Parte F — Diário de Debate (para entregar)

Respondam, em texto, nesta célula (ou em uma célula de markdown própria de cada integrante), conectando o que construíram acima aos conceitos da aula:

1. **Interface uniforme.** No nosso mini-roteador, o mesmo conjunto de métodos (GET, POST, PUT, PATCH, DELETE) se aplica a qualquer recurso registrado. Comparem isso com o endpoint único `/api` da Parte E (Nível 0) — qual das duas abordagens é mais fácil de estender com um novo recurso, e por quê?

2. **Status codes e a interface uniforme.** No endpoint de Nível 0, um erro de "pedido não encontrado" retorna status 200 OK, com o erro escondido dentro do corpo. Por que isso é um problema prático, além de estético? Pensem em como uma biblioteca cliente genérica (que não conhece o formato específico dessa API) reagiria a cada caso.

3. **Idempotência na prática.** Depois de rodar os experimentos da Parte D, expliquem com suas palavras por que PUT e DELETE podem ser reenviados com segurança por uma camada de retry automático, mas POST não pode.

4. **Gancho para a Aula 3.** Nossa API da Parte C tem endpoints, métodos e status codes bem definidos — mas nenhum contrato formal documentado. Se vocês tivessem que entregar essa API para outro time consumir, sem acesso ao nosso código-fonte, que informações vocês sentem falta que só um contrato formal (o assunto da próxima aula) resolveria?

> Registrem, se usaram alguma IA para ajudar a entender ou depurar o exercício desta prática, no que ela ajudou e por que a explicação fez sentido — igual combinamos para os notebooks de todas as disciplinas.

## Parte G — Diário de Debugging (para entregar)

Para **cada** uma das falhas que encontrarem, preencham um bloco como o modelo abaixo (copiem e repitam). Não precisam encontrar exatamente 5 — documentem quantas encontrarem, mas o notebook original tem 5.

---

**Falha #___**

- **Onde estava:** (nome da função/célula)
- **O que a célula deveria fazer:**
- **O que ela fazia de errado:**
- **Como perceberam:** (o que no output chamou atenção)
- **Como a IA ajudou a diagnosticar:** (o que vocês perguntaram, o que ela sugeriu)
- **A correção:** (trecho de código corrigido)
- **Por que essa correção é a certa** (não só "a IA disse" — expliquem com suas palavras):

---

**Falha #1**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa:

**Falha #2**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa:

**Falha #3**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa:

**Falha #4**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa:

**Falha #5**

- Onde estava:
- O que deveria fazer:
- O que fazia de errado:
- Como perceberam:
- Como a IA ajudou:
- Correção:
- Por que está certa: